# Final evaluation and error analysis

The models are compared once on the official test set. This notebook saves the complete metrics, predictions, per-class results, confusion matrix, and the most common confusion pairs.

## Course connection

The evaluation combines the classification metrics and error-analysis practices used throughout the course with the project requirement to explain which banking intents remain difficult.

In [ ]:
# Use the official test split only now, after model selection is complete.
import sys
from pathlib import Path
root = Path.cwd().parent
sys.path.insert(0, str(root))
import gc, json, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from src.data.load_data import load_banking77
from src.models.bilstm import build_vocab, collate_batch, BiLSTMClassifier
from src.models.transformer import MODEL_NAME
from src.evaluation.evaluate import classification_metrics
from src.utils import set_seed, get_device
set_seed(42)
device = get_device(); dataset = load_banking77(validation_size=0.1, seed=42); test = dataset["test"]
texts, labels = test["text"], test["label"]
rows = []; predictions = {}

In [ ]:
# E0 is a lexical reference point for the neural models.
# E0: TF-IDF + Logistic Regression
train = dataset["train"]
model = Pipeline([ ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)), ("classifier", LogisticRegression(max_iter=1000)) ])
model.fit(train["text"], train["label"]); pred = model.predict(texts); predictions["E0_tfidf"] = pred.tolist(); rows.append({"experiment":"E0_tfidf", **classification_metrics(labels, pred)})

In [ ]:
# Reload the best BiLSTM checkpoint and evaluate it on exactly the same queries.
# E1: trained BiLSTM
vocab = build_vocab(train["text"]); loader = DataLoader(list(zip(texts, labels)), batch_size=64, shuffle=False, collate_fn=lambda b: collate_batch(b, vocab)); bilstm = BiLSTMClassifier(len(vocab), num_classes=77).to(device); bilstm.load_state_dict(torch.load(root / "results/bilstm_best.pt", map_location=device)); bilstm.eval(); pred=[]
with torch.no_grad():
    for inputs, lengths, _ in loader: pred.extend(bilstm(inputs.to(device), lengths.to(device)).argmax(1).cpu().tolist())
predictions["E1_bilstm"] = pred; rows.append({"experiment":"E1_bilstm", **classification_metrics(labels, pred)})

In [ ]:
# This helper keeps tokenization and padding identical to the Transformer runs.
def transformer_predictions(checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(root / checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(root / checkpoint).to(device).eval()
    tokenized = dataset["test"].map(lambda b: tokenizer(b["text"], truncation=True, max_length=64), batched=True).remove_columns(["text"])
    tokenized.set_format("torch"); collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    loader = DataLoader(tokenized, batch_size=32, shuffle=False, collate_fn=collator); out=[]
    with torch.no_grad():
        for batch in loader:
            batch={k:v.to(device) for k,v in batch.items()}; out.extend(model(**batch).logits.argmax(1).cpu().tolist())
    del model, tokenizer, loader; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return out
for name, checkpoint in [("E2_frozen_distilbert", "results/04_distilbert_frozen_checkpoint"), ("E3_finetuned_distilbert", "results/05_distilbert_finetuning_checkpoint"), ("E4_lr_5e-5", "results/06_experiments_lr_5e-5_checkpoint")]:
    pred=transformer_predictions(checkpoint); predictions[name]=pred; rows.append({"experiment":name, **classification_metrics(labels, pred)})

In [ ]:
# Save both aggregate and class-level evidence so conclusions can be checked later.
metrics = pd.DataFrame(rows); Path(root / "results").mkdir(exist_ok=True); metrics.to_csv(root / "results/metrics.csv", index=False); pd.DataFrame(predictions).assign(text=texts, label=labels).to_csv(root / "results/predictions/test_predictions.csv", index=False); print(metrics.to_string(index=False))
# Save per-class details and the full confusion matrix for the best validation configuration.
best_name = "E3_finetuned_distilbert"; report = classification_report(labels, predictions[best_name], output_dict=True, zero_division=0); pd.DataFrame(report).T.to_csv(root / "results/per_class_metrics.csv"); np.savetxt(root / "results/confusion_matrix.csv", confusion_matrix(labels, predictions[best_name]), delimiter=",", fmt="%d")